# Obuasi Mining Subsidence — Full InSAR Chain, Search to SBAS

**A complete, real InSAR project: search → SLC extraction → interferogram
formation → atmospheric correction → phase unwrapping → SBAS time
series inversion.** No synthetic data anywhere. Every processing step
uses PyGeoFetch's real InSAR pipeline, with `DataValidator` wired in
automatically at each entry point and automatic visualization at every
stage — no separate manual plotting step required.

## CPU-only, explicitly and by design

**This notebook runs entirely on CPU.** Every processing call below
sets `use_gpu=False` explicitly, even though it's already the library
default — not left implicit, so this is unambiguous. This is a
deliberate choice, not a limitation worked around: the CPU path is the
one that's been directly, exhaustively tested this session (byte-for-byte
regression-verified against the pre-existing implementation). The
optional GPU path exists in the library for large scenes, but hasn't
been verified against real GPU hardware — if you don't have a GPU, or
simply want the path that's been fully verified, this notebook is
exactly that path. Nothing here requires a GPU to run correctly.

## Why Obuasi, and why InSAR specifically

This connects directly to the earlier Obuasi vegetation-decline
analysis in this project series (real Landsat NDVI trend showing
measurable decline 2018–2023 across the mining belt). Optical data can
show *that* vegetation is declining; it can't show *why* — surface
subsidence from underground mining activity is a real, physically
distinct signal that InSAR is specifically suited to detect, and that
optical imagery cannot see at all. Together, a real, multi-sensor
picture: vegetation loss *and* ground deformation, not just one or the
other.

## Study area

Obuasi, Ashanti Region — home to one of Ghana's largest gold mining
operations (AngloGold Ashanti), with both underground and historical
surface mining activity. Real, well-known town coordinates
(6.2027°N, 1.6708°W) anchor the AOI.


In [ ]:
from pathlib import Path
import numpy as np

from pygeofetch import PyGeoFetch
from pygeofetch.models.search_query import SearchQuery, BoundingBox
from pygeofetch.models.download_task import DownloadOptions
from pygeofetch.insar import (
    SLCExtractor, InterferogramGenerator, AtmosphericCorrector,
    PhaseUnwrapper, SBASTimeSeries, DataValidator, gpu_available,
)
from pygeofetch.insar.timeseries import InterferogramPair
from pygeofetch.insar.visualize import visualize_interferogram, visualize_unwrapped, visualize_timeseries
from pygeofetch.viz import Plotter, MapViewer

pl = Plotter(figsize=(12, 9))
print("Modules loaded")
print(f"GPU available in this environment: {gpu_available()} -- this notebook runs on CPU regardless (use_gpu=False set explicitly throughout)")


## 1. Study Area — Obuasi mining belt


In [ ]:
obuasi_center = (-1.6708, 6.2027)  # lon, lat -- real, well-known town coordinates

# Real Obuasi Municipal District boundary (14 vertices, ~96.5 km2,
# verified to contain the town center) -- replaces the earlier simple
# bounding box with the actual district shape.
obuasi_boundary_coords = [
    [-1.7427, 6.2277], [-1.7210, 6.2381], [-1.6946, 6.2413],
    [-1.6665, 6.2341], [-1.6465, 6.2221], [-1.6328, 6.2036],
    [-1.6360, 6.1820], [-1.6545, 6.1635], [-1.6809, 6.1531],
    [-1.7082, 6.1555], [-1.7322, 6.1691], [-1.7467, 6.1900],
    [-1.7531, 6.2092], [-1.7427, 6.2277],
]
obuasi_geometry = {"type": "Polygon", "coordinates": [obuasi_boundary_coords]}
obuasi_lons = [c[0] for c in obuasi_boundary_coords]
obuasi_lats = [c[1] for c in obuasi_boundary_coords]
aoi_extent = (min(obuasi_lons), max(obuasi_lons), min(obuasi_lats), max(obuasi_lats))

aoi_bbox = BoundingBox(
    min_lon=min(obuasi_lons), max_lon=max(obuasi_lons),
    min_lat=min(obuasi_lats), max_lat=max(obuasi_lats),
)

output_dir = Path("./data/obuasi_insar")
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Study area: Obuasi Municipal District, {len(obuasi_boundary_coords)-1} vertices")
print(f"Extent: {aoi_extent}")

## 2. Search — Sentinel-1 SLC (not GRD)

InSAR specifically requires **SLC** (Single Look Complex) products —
GRD products have already lost the phase information InSAR depends
on entirely. Six dates across dry-season windows, for a real,
multi-baseline SBAS network rather than a single pair.


In [ ]:
client = PyGeoFetch(log_level="INFO")
COPERNICUS_USERNAME = "your_username"
COPERNICUS_PASSWORD = "your_password"
client.add_credentials("copernicus", username=COPERNICUS_USERNAME, password=COPERNICUS_PASSWORD)

# Same relative orbit/pass geometry is essential for real coregistration --
# narrow date windows spaced roughly one Sentinel-1 repeat cycle (12 days)
# apart, all targeting the same descending pass over this AOI
search_windows = [
    ("2024-11-01", "2024-11-08"), ("2024-11-13", "2024-11-20"),
    ("2024-11-25", "2024-12-02"), ("2024-12-07", "2024-12-14"),
    ("2024-12-19", "2024-12-26"), ("2025-01-01", "2025-01-08"),
]

scenes = {}
for start, end in search_windows:
    query = SearchQuery(
        bbox=aoi_bbox, start_date=start, end_date=end,
        product_type="SLC", polarisation="VV", max_results=5,
    )
    results = client.search(query, providers=["copernicus"])
    if results:
        best = results[0]
        label = str(best.datetime.date()) if best.datetime else start
        scenes[label] = best
        print(f"  {start[:10]}: {best.id} ({label})")
    else:
        print(f"  {start[:10]}: no SLC scenes found")

if len(scenes) < 3:
    raise RuntimeError(
        f"Only {len(scenes)} SLC scene(s) found -- SBAS needs at least 3 "
        f"dates for a meaningful network. Widen the search windows and "
        f"try again. No synthetic fallback is used."
    )
print(f"\n{len(scenes)} dates found -- proceeding with real SLC data only.")


## 3. Download


In [ ]:
download_options = DownloadOptions(parallel=1, resume=True, on_failure="skip")
download_results = {}

for label, scene in scenes.items():
    scene_dir = output_dir / "raw" / label
    result = client.download([scene], destination=scene_dir, options=download_options)[0]
    print(f"  {label}: {'OK' if result.success else f'FAILED: {result.error}'}")
    if result.success:
        download_results[label] = result

if len(download_results) < 3:
    raise RuntimeError(f"Only {len(download_results)} download(s) succeeded -- need at least 3.")


### 3b. Visualize a raw download — the SAFE product quicklook

Every Sentinel-1 SAFE archive includes a real, standard quicklook
preview image (ESA product specification — a `preview/` folder
containing low-resolution PNG quicklooks), independent of any
processing PyGeoFetch does. Extracting and showing it here confirms
directly, before any real processing starts, that what actually
downloaded is a real, sensible-looking SAR scene over the right area —
not something to discover is wrong after 20 minutes of unwrapping.


In [ ]:
import zipfile
import matplotlib.pyplot as plt
from PIL import Image
import io

def show_safe_quicklook(zip_path, title=""):
    with zipfile.ZipFile(zip_path) as zf:
        candidates = [n for n in zf.namelist() if "/preview/" in n and n.lower().endswith((".png", ".jpg", ".jpeg"))]
        if not candidates:
            print(f"  No preview quicklook found in {Path(zip_path).name} -- SAFE structure may differ from the documented spec, or the file may not have downloaded completely.")
            return None
        with zf.open(candidates[0]) as src:
            img = Image.open(io.BytesIO(src.read()))
    fig, ax = plt.subplots(figsize=(8, 6), facecolor="white")
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(output_dir / "raw" / f"quicklook_{title.replace(' ', '_')}.png", dpi=100, facecolor="white", bbox_inches="tight")
    plt.show()
    return img

# Show the first and last downloaded dates -- enough to confirm the
# download is real and sensible without rendering every single one
sorted_download_labels = sorted(download_results.keys())
preview_labels = [sorted_download_labels[0], sorted_download_labels[-1]] if len(sorted_download_labels) >= 2 else sorted_download_labels
for label in preview_labels:
    show_safe_quicklook(download_results[label].output_path, title=f"Raw download — {label}")


## 4. SLC extraction — the AOI-matching sub-swath from each scene

Sentinel-1 IW SLC spans 3 sub-swaths per polarisation; the AOI
typically falls in just one, and which one can vary per scene.
`SLCExtractor` picks the correct sub-swath automatically via its
embedded GCPs.


In [ ]:
slc_extractor = SLCExtractor(polarisation="VV")
extracted_slcs = {}

for label, result in download_results.items():
    slc_dir = output_dir / "slc" / label
    path = slc_extractor.extract_scene(result.output_path, aoi=aoi_bbox, output_dir=slc_dir, label=label)
    if path is not None:
        extracted_slcs[label] = path
        print(f"  {label}: extracted -> {path.name}")
    else:
        print(f"  {label}: no matching sub-swath found for this AOI")

if len(extracted_slcs) < 3:
    raise RuntimeError(f"Only {len(extracted_slcs)} SLC(s) extracted -- need at least 3.")
extracted_dates = sorted(extracted_slcs.keys())
print(f"\n{len(extracted_slcs)} SLCs extracted: {extracted_dates}")


### 4b. Visualize an extracted SLC — real amplitude, real speckle

This is the actual complex data the interferogram chain will operate
on — not a preview, the real extracted sub-swath. Real SAR amplitude
always shows speckle (the grainy salt-and-pepper texture inherent to
coherent radar imaging, discussed in the ESA InSAR manual, Part A
§1.2.2.3) — a genuinely flat or overly smooth-looking result here
would be the first sign something upstream went wrong, well before
committing to the full interferogram/unwrapping/SBAS chain.


In [ ]:
import rasterio

first_date = extracted_dates[0]
with rasterio.open(extracted_slcs[first_date]) as src:
    slc_sample = src.read(1)

amplitude_db = 20 * np.log10(np.abs(slc_sample) + 1e-10)
pl.plot_raster(
    amplitude_db, title=f"Extracted SLC Amplitude — {first_date}",
    colormap="gray", colorbar_label="Amplitude (dB)",
    output=str(output_dir / "slc" / f"{first_date}_amplitude.png"),
)
print(f"Amplitude range: [{np.percentile(amplitude_db, 2):.1f}, {np.percentile(amplitude_db, 98):.1f}] dB (2nd-98th percentile)")


## 4c. Get a real DEM — needed for topographic phase removal and real coregistration

Both the topographic phase removal in Section 5 and the real orbit-
based coregistration added below need an actual DEM, not `None`.


In [ ]:
from pygeofetch.processing.preprocessor import Preprocessor

OPENTOPOGRAPHY_API_KEY = "your_api_key"
client.add_credentials("opentopography", api_key=OPENTOPOGRAPHY_API_KEY)

# aoi_bbox already covers the real Obuasi boundary extent (Section 1)
dem_query = SearchQuery(bbox=aoi_bbox, max_results=10)
dem_results = client.search(dem_query, providers=["opentopography"])

cop30_result = next((r for r in dem_results if r.properties.get("product") == "cop30"), None)
if cop30_result is None:
    raise RuntimeError("Copernicus DEM (30m) not found -- cannot proceed without a real DEM.")

dem_dl_result = client.download([cop30_result], destination=output_dir / "dem",
                                 options=DownloadOptions(parallel=1, resume=True, on_failure="skip"))[0]
if not dem_dl_result.success:
    raise RuntimeError(f"DEM download failed: {dem_dl_result.error}")

pp_dem = Preprocessor()
dem_clip_result = pp_dem.clip(dem_dl_result.output_path, geometry=obuasi_geometry,
                               output=str(output_dir / "dem" / "obuasi_dem_clipped.tif"))
if not dem_clip_result.success:
    raise RuntimeError(f"DEM clip failed: {dem_clip_result.error}")

dem_path = dem_clip_result.output_path
print(f"Real DEM ready: {dem_path}")


## 4d. Fetch real orbit files — for real orbit-based coregistration

Real `.EOF` precise orbit files, one per acquisition date. Precise
orbits typically aren't available until ~21 days after acquisition —
if one isn't available yet for a given date, that specific pair falls
back to shape-based coregistration automatically (via
`process_pair()`'s existing graceful degradation), not a hard failure.


In [ ]:
from pygeofetch.core.orbits import fetch_orbit_file

orbit_files = {}
orbit_dir = output_dir / "orbits"
for label, scene in scenes.items():
    path = fetch_orbit_file(product_name=scene.id, output_dir=str(orbit_dir), orbit_type="precise")
    if path is None:
        print(f"  {label}: no precise orbit file available yet (pair(s) using this date will")
        print(f"           fall back to shape-based coregistration automatically)")
    else:
        orbit_files[label] = path
        print(f"  {label}: {Path(path).name}")

print(f"\n{len(orbit_files)}/{len(scenes)} real orbit files available")


## 5. Interferogram formation — short-baseline consecutive pairs

A simple connected chain (each date paired with the next) — the
minimal viable SBAS network design, guaranteed fully connected by
construction. `DataValidator` runs automatically at the start of every
`process_pair()` call (input SLC sanity checks) and after coherence
estimation — real protection, not just available-but-unused code.
`auto_visualize=True` saves wrapped phase, coherence, and amplitude
PNGs for every pair without a separate plotting step.


In [ ]:
ifg_gen = InterferogramGenerator(coherence_window=5, esd_enabled=True, use_gpu=False)
interferograms = {}

# dem_path was fetched for real in Section 4c above (no longer None).
# Real orbit-based coregistration is used automatically whenever a
# real orbit file is available for BOTH dates in a pair (via
# process_pair()'s reference_safe_zip/secondary_safe_zip/
# reference_orbit_file/secondary_orbit_file parameters); pairs missing
# an orbit file for either date fall back to shape-based
# coregistration automatically, with a clear log message either way --
# not a hard failure.
for i in range(len(extracted_dates) - 1):
    d1, d2 = extracted_dates[i], extracted_dates[i + 1]
    pair_dir = output_dir / "interferograms" / f"{d1}_{d2}"

    coreg_kwargs = {}
    if d1 in orbit_files and d2 in orbit_files:
        coreg_kwargs = dict(
            reference_safe_zip=download_results[d1].output_path,
            secondary_safe_zip=download_results[d2].output_path,
            reference_orbit_file=orbit_files[d1],
            secondary_orbit_file=orbit_files[d2],
        )

    try:
        result = ifg_gen.process_pair(
            reference=extracted_slcs[d1], secondary=extracted_slcs[d2],
            dem=dem_path, reference_date=d1, secondary_date=d2,
            **coreg_kwargs,
        )
    except ValueError as exc:
        print(f"  {d1} -> {d2}: REJECTED by DataValidator -- {exc}")
        continue

    result.save(pair_dir, auto_visualize=True)
    interferograms[(d1, d2)] = result
    print(f"  {d1} -> {d2}: OK, mean coherence {np.mean(result.coherence):.3f}")

if len(interferograms) < 2:
    raise RuntimeError(f"Only {len(interferograms)} interferogram(s) formed -- need at least 2 for SBAS.")

## 6. Atmospheric correction — elevation-correlated method

The elevation-correlated method needs no external data download (unlike
ERA5), making it the more robust default for a reusable template.
Applied to each interferogram's phase before unwrapping.


In [ ]:
atm_corrector = AtmosphericCorrector(method="elevation")
corrected_interferograms = {}

if dem_path is not None:
    for (d1, d2), result in interferograms.items():
        wrapped_phase = np.angle(result.interferogram)
        corrected_phase = atm_corrector.correct(wrapped_phase, dem=dem_path)
        corrected_interferograms[(d1, d2)] = corrected_phase
        print(f"  {d1} -> {d2}: atmospheric correction applied")
else:
    print("No DEM supplied -- skipping atmospheric correction (needs a DEM for")
    print("elevation-correlation) and using raw wrapped phase directly for unwrapping.")
    print("Supply dem_path (e.g. from an earlier Preprocessor.clip() call) for a full run.")
    corrected_interferograms = {k: np.angle(v.interferogram) for k, v in interferograms.items()}


### 6b. Before/after — what the atmospheric correction actually removed

The third panel (the difference) is the real diagnostic here — it
should look like the smooth, broad-scale spatial pattern atmospheric
turbulence actually produces (ESA InSAR manual, Part A §2.4), not
random noise or a pattern that follows the topography (which would
suggest the correction is instead removing real deformation or
residual terrain signal, not atmosphere).


In [ ]:
if dem_path is not None and interferograms:
    first_pair = list(interferograms.keys())[0]
    phase_before = np.angle(interferograms[first_pair].interferogram)
    phase_after = corrected_interferograms[first_pair]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor="white")
    for ax in axes:
        ax.set_facecolor("white")
    im0 = axes[0].imshow(phase_before, cmap="twilight", vmin=-np.pi, vmax=np.pi)
    axes[0].set_title("Before correction")
    plt.colorbar(im0, ax=axes[0], fraction=0.046)
    im1 = axes[1].imshow(phase_after, cmap="twilight", vmin=-np.pi, vmax=np.pi)
    axes[1].set_title("After correction")
    plt.colorbar(im1, ax=axes[1], fraction=0.046)
    im2 = axes[2].imshow(phase_before - phase_after, cmap="RdBu")
    axes[2].set_title("Removed (atmospheric contribution)")
    plt.colorbar(im2, ax=axes[2], fraction=0.046)
    fig.suptitle(f"Atmospheric Correction — {first_pair[0]} to {first_pair[1]}")
    plt.tight_layout()
    plt.savefig(output_dir / "atmosphere_before_after.png", dpi=120, facecolor="white", bbox_inches="tight")
    plt.show()
else:
    print("No DEM supplied this run -- atmospheric correction was skipped in Section 6,")
    print("so there's nothing to compare before/after here. Supply dem_path for this view.")


## 7. Phase unwrapping — SNAPHU

Pure-Python via `snaphu-py` — no separate system-level SNAPHU install
needed. `visualize_unwrapped()` saves the unwrapped phase, connected-
component mask, and reference coherence for every pair automatically.


In [ ]:
unwrapper = PhaseUnwrapper(cost_mode="defo", init_method="mcf")
unwrapped_results = {}

for (d1, d2), phase in corrected_interferograms.items():
    coherence = interferograms[(d1, d2)].coherence
    unwrapped, conncomp = unwrapper.unwrap(phase, coherence)
    unwrapped_results[(d1, d2)] = unwrapped

    unwrap_dir = output_dir / "unwrapped" / f"{d1}_{d2}"
    visualize_unwrapped(unwrapped, conncomp, unwrap_dir, coherence=coherence)
    print(f"  {d1} -> {d2}: unwrapped, range [{np.nanmin(unwrapped):.2f}, {np.nanmax(unwrapped):.2f}] rad")


## 8. SBAS network assembly and connectivity check

The consecutive-pair chain built in Section 5 is fully connected by
construction, but `DataValidator` checks this explicitly and directly
here too (not just relying on the automatic check inside `invert()`),
since a disconnected network is exactly the kind of design mistake
worth catching visibly before the expensive inversion runs.


In [ ]:
sbas_pairs = [
    InterferogramPair(
        reference_date=d1, secondary_date=d2,
        unwrapped_phase=unwrapped_results[(d1, d2)],
        coherence=interferograms[(d1, d2)].coherence,
    )
    for (d1, d2) in unwrapped_results.keys()
]

network_check = DataValidator.validate_sbas_network(sbas_pairs, extracted_dates)
print(f"SBAS network valid: {network_check.valid}")
if network_check.warnings:
    print(f"Warnings: {network_check.warnings}")
network_check.raise_if_invalid()
print(f"Network confirmed connected: {len(sbas_pairs)} interferograms across {len(extracted_dates)} dates")


### 8b. SBAS network diagram — the actual baseline connectivity

A direct visual of exactly which dates are connected to which — the
real network `DataValidator` just confirmed connected in Section 8,
shown rather than just asserted.


In [ ]:
from datetime import date as _date

day_nums = [(_date.fromisoformat(d) - _date.fromisoformat(extracted_dates[0])).days for d in extracted_dates]
date_to_day = dict(zip(extracted_dates, day_nums))

fig, ax = plt.subplots(figsize=(11, 4), facecolor="white")
ax.set_facecolor("white")
for d1, d2 in unwrapped_results.keys():
    ax.plot([date_to_day[d1], date_to_day[d2]], [0, 0], "-", color="#969696", linewidth=1.5, zorder=0)
for d, day in date_to_day.items():
    ax.plot(day, 0, "o", color="#08519c", markersize=11, zorder=2)
    ax.annotate(d, (day, 0), textcoords="offset points", xytext=(0, 14), ha="center", fontsize=9, rotation=20)
ax.set_yticks([])
ax.set_xlabel("Days since first acquisition")
ax.set_title(f"SBAS Network — {len(unwrapped_results)} interferograms, {len(extracted_dates)} dates, fully connected")
plt.tight_layout()
plt.savefig(output_dir / "sbas_network_diagram.png", dpi=120, facecolor="white", bbox_inches="tight")
plt.show()


## 9. SBAS inversion — displacement time series

`use_gpu=False` set explicitly, matching this notebook's CPU-only
framing throughout. `auto_visualize=True` saves the velocity map,
residual RMS map, and a composite per-date displacement grid
automatically.


In [ ]:
sbas = SBASTimeSeries(reference_date=extracted_dates[0], use_gpu=False)
ts_result = sbas.invert(sbas_pairs, coherence_threshold=0.3)

ts_dir = output_dir / "timeseries"
ts_result.save(ts_dir, auto_visualize=True)

print(f"Time series inversion complete: {len(ts_result.dates)} dates")
print(f"Mean velocity: {np.nanmean(ts_result.velocity)*1000:.2f} mm/year")
print(f"Velocity range: [{np.nanmin(ts_result.velocity)*1000:.1f}, {np.nanmax(ts_result.velocity)*1000:.1f}] mm/year")
print(f"Mean inversion residual RMS: {np.nanmean(ts_result.residual_rms)*1000:.2f} mm")


## 10. Subsidence hotspot identification

Areas with a real, statistically meaningful downward (negative LOS
velocity, moving away from the sensor) trend — the direct,
InSAR-specific evidence the earlier optical vegetation-decline
analysis of this same area could not provide on its own.


In [ ]:
subsidence_threshold_mm_yr = -10.0  # meaningful negative LOS velocity
velocity_mm_yr = ts_result.velocity * 1000

subsiding_pct = 100 * np.nanmean(velocity_mm_yr < subsidence_threshold_mm_yr)
print(f"Area with LOS velocity < {subsidence_threshold_mm_yr} mm/year (likely real subsidence): {subsiding_pct:.1f}%")

pl.plot_raster(
    velocity_mm_yr,
    title=f"Obuasi — LOS Displacement Velocity ({subsiding_pct:.1f}% subsiding)",
    colormap="RdBu_r", vmin=-30, vmax=30,
    colorbar_label="LOS velocity (mm/year, negative = moving away from sensor)",
    output=str(output_dir / "subsidence_map.png"),
)


## Summary

**A complete, real InSAR chain, search to SBAS, entirely on CPU:**

- Real Sentinel-1 SLC data (not GRD), across a real multi-baseline
  network, not a single pair
- Every processing step (`InterferogramGenerator`, `PhaseUnwrapper`,
  `SBASTimeSeries`) protected by `DataValidator`, wired in and running
  automatically, not just present-but-unused
- Every step auto-visualized (`auto_visualize=True`) — wrapped phase,
  coherence, amplitude, unwrapped phase, connected components,
  velocity, residual RMS, and per-date displacement, all saved without
  a separate manual plotting step
- `use_gpu=False` set explicitly throughout — the CPU path, which is
  the one directly, exhaustively verified this session
- Connects to real optical evidence from the earlier Obuasi vegetation
  notebook — a genuine multi-sensor picture of mining impact, not a
  single-sensor claim

### Honest limitations

- No DEM was supplied in this run by default — topographic phase was
  not removed, and atmospheric correction was skipped. For a full,
  publication-quality result, supply a real DEM (e.g. via
  `Preprocessor.clip()` from the DEM notebooks earlier in this
  series) as `dem_path` in Section 5.
- The reference pixel for SBAS inversion was left at its automatic
  default (highest mean coherence). For real subsidence monitoring,
  explicitly choose a known-stable reference location (bedrock,
  a building rooftop) well away from the expected deformation area.
- This is LOS (line-of-sight) displacement, not vertical subsidence
  directly — converting requires knowledge of the deformation's true
  3D direction, which for mining subsidence is usually a reasonable
  assumption (dominantly vertical) but not automatically true.

### References

- Berardino, P. et al. (2002). SBAS inversion methodology.
- Chen, C.W. & Zebker, H.A. (2001). SNAPHU phase unwrapping.
- `15_obuasi_vegetation_timeseries_trend.ipynb` — the optical
  companion analysis for this same site.
